# Building a Customer Support StateGraph Agent (LLM Integration)

Welcome! This Jupyter Notebook is a tutorial designed to help you learn how to build state machines (specifically, a **StateGraph**) using LangGraph, powered by an LLM (Language Model) from Hugging Face Hub.

## Business Scenario
A customer support bot must handle incoming messages and decide whether to:
1. **Answer FAQs** (informational requests).
2. **Log complaints** (unhappy/dissatisfied customers).
3. **Troubleshoot Technical Support** (technical errors, setup).
4. **Escalate to an agent** (complex technical or account requests).

## StateGraph Design
Our graph starts with a `classify_query` node to categorize the query. Depending on the classification, it uses a **conditional edge** to route to one of the following:
* `faq_node` (informational)
* `complaint_node` (dissatisfied)
* `technical_support_node` (technical issues)
* `escalate_node` (complex/human request)

Let's get started by loading env vars and initializing the Hugging Face LLM.

In [1]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# Load environment variables from .env
load_dotenv()

# Initialize the Chat Model Pipeline using meta-llama/Meta-Llama-3-8B-Instruct
# via the Hugging Face Hub endpoint.
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if hf_token:
    llm_endpoint = HuggingFaceEndpoint(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
        temperature=0.3,
        provider="auto",
        huggingfacehub_api_token=hf_token
    )
    chat_model = ChatHuggingFace(llm=llm_endpoint)
    print("Chat model successfully initialized using HUGGINGFACEHUB_API_TOKEN!")
else:
    raise ValueError("HUGGINGFACEHUB_API_TOKEN not found in environment. Please set it in your .env file.")

Chat model successfully initialized using HUGGINGFACEHUB_API_TOKEN!


/Users/aaditya/Desktop/simple-agents/customer_support/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Defining the State Schema

In LangGraph, the core of any agent is its state. The state is represented by a Python dictionary or `TypedDict` and is passed from node to node. Every node can modify or add keys to this state.

For our customer support bot, we define `BotState` with three keys:
* `query`: The initial user message.
* `type`: The classified category (`faq`, `complaint`, `technical_support`, or `escalate`).
* `response`: The final response produced by the handling nodes.

In [2]:
class BotState(TypedDict):
    """
    Represents the state of our customer support graph.
    
    Keys:
        query (str): The initial user query or message.
        type (str): The categorized type of the query ('faq', 'complaint', 'technical_support', or 'escalate').
        response (str): The generated response from the appropriate handling node.
    """
    query: str
    type: str
    response: str

## 2. Defining Node Functions

Nodes represent execution steps. In LangGraph, nodes are standard Python functions that receive the current state as an input and return a dictionary containing updates to the state.

In [3]:
# Node 1: Multi-Intent Classification Node
def classify_query(state: BotState) -> dict:
    """Categorizes the user's message into one of four distinct intents."""
    query = state["query"]
    
    messages = [
        SystemMessage(content=(
            "You are an advanced routing bot for customer operations. "
            "Classify the user message into exactly one of these lowercase words: "
            "'faq', 'complaint', 'technical_support', or 'escalate'.\n\n"
            "Classification Rules:\n"
            "- 'faq': Simple, general business inquiries (hours, returns, tracking, shipping rules).\n"
            "- 'complaint': Expressing frustration, bugs, delays, anger, or demanding a refund.\n"
            "- 'technical_support': Specific issues regarding hardware, software setup, code, errors, or configuration.\n"
            "- 'escalate': High-value corporate requests, legal threats, or explicitly asking for a human manager.\n\n"
            "Respond with ONLY the single word category name. No extra text, punctuation, or preamble."
        )),
        HumanMessage(content=f"Message: {query}")
    ]
    
    response = chat_model.invoke(messages)
    category = response.content.strip().lower()
    
    # Strict validation mapping layer
    valid_intents = {"faq", "complaint", "technical_support", "escalate"}
    if category not in valid_intents:
        # Check substring fallbacks if the LLM added filler words
        for intent in valid_intents:
            if intent in category:
                category = intent
                break
        else:
            category = "escalate"  # Default safety fallback
        
    print(f"[Node: classify_query] -> Intent Classified as: '{category}'")
    return {"type": category}

In [4]:
# Node 2: Dynamic FAQ Node
def faq_node(state: BotState) -> dict:
    """Generates a dynamic answer based on standard store parameters."""
    query = state["query"]
    messages = [
        SystemMessage(content=(
            "You are a helpful customer support bot answering FAQs. Answer the user's question accurately based on these facts:\n"
            "- Shipping: 3-5 business days domestically, 10-14 days internationally.\n"
            "- Returns: 30-day window, items must be unopened.\n"
            "- Store Hours: 9 AM - 6 PM EST, Mon-Fri.\n"
            "Keep the reply professional, polite, and concise (under 2 sentences)."
        )),
        HumanMessage(content=query)
    ]
    response = chat_model.invoke(messages)
    return {"response": response.content.strip()}

In [5]:
# Node 3: Dynamic Complaint Log & De-escalation Node
def complaint_node(state: BotState) -> dict:
    """Logs the issue and generates an empathetic, professional apology."""
    query = state["query"]
    messages = [
        SystemMessage(content=(
            "You are an empathetic customer recovery specialist. "
            "Acknowledge the user's frustration, apologize sincerely, and inform them that an incident ticket "
            "has been logged with our QA team for immediate investigation. Keep it reassuring and short."
        )),
        HumanMessage(content=query)
    ]
    response = chat_model.invoke(messages)
    return {"response": response.content.strip()}

In [6]:
# Node 4: Dynamic Technical Troubleshooting Node
def technical_support_node(state: BotState) -> dict:
    """Provides targeted, structured, step-by-step guidance for technical issues."""
    query = state["query"]
    messages = [
        SystemMessage(content=(
            "You are an expert IT Tier-1 technical support engineer. Provide the user with 2 clear, bulleted troubleshooting "
            "steps to address their technical problem based on their message. Keep the tone helpful and technical."
        )),
        HumanMessage(content=query)
    ]
    response = chat_model.invoke(messages)
    return {"response": response.content.strip()}

In [7]:
# Node 5: Human Escalation Node
def escalate_node(state: BotState) -> dict:
    """Prepares standard handoff protocols."""
    return {"response": "This request requires executive routing or direct account management. I am transferring this interaction immediately to a senior agent. Live chat queue time is currently < 2 minutes."}

## 3. Defining the Routing Logic (Conditional Edge)

In LangGraph, conditional routing is defined by a routing function. This function reads the state and returns a string indicating which node should be executed next.

In [8]:
# Define Conditional Routing Logic
def route_based_on_type(state: BotState) -> str:
    """Maps state['type'] exactly to the string name of the next Graph node."""
    mapping = {
        "faq": "faq_node",
        "complaint": "complaint_node",
        "technical_support": "technical_support_node",
        "escalate": "escalate_node"
    }
    return mapping.get(state["type"], "escalate_node")

## 4. Graph Construction and Compilation

Now, we will assemble our nodes and edges into a `StateGraph`, then compile it.

In [9]:
# Build and Compile the Workflow StateGraph
workflow = StateGraph(BotState)

# Register the nodes
workflow.add_node("classify_query", classify_query)
workflow.add_node("faq_node", faq_node)
workflow.add_node("complaint_node", complaint_node)
workflow.add_node("technical_support_node", technical_support_node)
workflow.add_node("escalate_node", escalate_node)

# Set up standard edge flow
workflow.add_edge(START, "classify_query")

# Set up conditional branching
workflow.add_conditional_edges(
    "classify_query",
    route_based_on_type,
    {
        "faq_node": "faq_node",
        "complaint_node": "complaint_node",
        "technical_support_node": "technical_support_node",
        "escalate_node": "escalate_node"
    }
)

# Close all endpoints to standard termination
workflow.add_edge("faq_node", END)
workflow.add_edge("complaint_node", END)
workflow.add_edge("technical_support_node", END)
workflow.add_edge("escalate_node", END)

app = workflow.compile()
print("StateGraph compiled successfully!")

StateGraph compiled successfully!


## 5. Testing and Execution

Let's test the graph with some queries, tracking both the classification results and final responses.

In [10]:
test_cases = [
    {"name": "Complaint Test", "query": "I ordered overnight shipping and it's been 4 days! I want my money back."},
    {"name": "FAQ Test", "query": "Can you tell me what time your support line closes on Fridays?"},
    {"name": "Technical Support Test", "query": "My application is crashing with a '401 unauthorized connection' token error whenever I push data."},
    {"name": "Escalation Test", "query": "Get me a human manager right away. Your bot is frustrating me and I will cancel our corporate license."}
]

for case in test_cases:
    print(f"\n--- {case['name']} ---")
    result = app.invoke({"query": case["query"]})
    print(f"Final Output Response:\n{result['response']}\n")


--- Complaint Test ---
[Node: classify_query] -> Intent Classified as: 'complaint'
Final Output Response:
I'm so sorry to hear that you're experiencing frustration with your order. I can imagine how upsetting it must be to not receive your package on time. I want to assure you that I'm here to help and make things right.

I've immediately logged an incident ticket with our QA team for investigation. They'll be looking into the delay and working to resolve the issue as soon as possible. I'll also make sure to keep you updated on the status of your order.

In the meantime, I'd like to offer you a gesture of goodwill for the inconvenience. Would you like me to provide a discount code for your next purchase or a refund for the shipping cost? Please let me know how I can make it right for you.


--- FAQ Test ---
[Node: classify_query] -> Intent Classified as: 'faq'
Final Output Response:
Our support line is available from 9 AM to 6 PM EST, Monday through Friday. The support line closes at 